In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pandas only supports SQLAlchemy.*")
warnings.filterwarnings("ignore", category=RuntimeWarning) 
import pymysql
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

# 设置绘图风格与中文字体
plt.rcParams['font.sans-serif'] = ['SimHei'] 
plt.rcParams['axes.unicode_minus'] = False
# 使用 Seaborn 的极简风格
sns.set_theme(style="white", font='SimHei')

def get_db_connection():
    """建立 StarRocks 数据库连接"""
    return pymysql.connect(
        host='192.168.144.101',
        port=9030,
        user='hadoop',
        password='hadoop',
        charset='utf8'
    )

def fetch_cities():
    """获取 DWS 表中有数据的城市列表"""
    conn = get_db_connection()
    # 路径：catalog.db.table
    query = "SELECT DISTINCT city FROM iceberg_catalog.ershoufang.dws_community_daily WHERE city IS NOT NULL ORDER BY city"
    try:
        df = pd.read_sql(query, conn)
        return df['city'].tolist()
    finally:
        conn.close()

def fetch_neighborhood_hotness_data(city_name):
    """
    获取指定城市的板块加权平均关注度。
    聚合逻辑：sum(avg_attention_count * house_count) / sum(house_count) 
    根据 Iceberg 元数据注释，聚合全部分区数据且不考虑 double counting。
    """
    conn = get_db_connection()
    # 这里的聚合逻辑保证了房源量大的小区对板块热度的贡献权重更大
    query = f"""
    SELECT 
        neighborhood,
        SUM(avg_attention_count * house_count) / SUM(house_count) as weighted_attention
    FROM iceberg_catalog.ershoufang.dws_community_daily
    WHERE city = '{city_name}'
        AND neighborhood IS NOT NULL AND neighborhood != ''
        AND house_count > 0 -- 避免除零异常
    GROUP BY neighborhood
    ORDER BY weighted_attention DESC
    LIMIT 30 -- 截取前 30 名进行展示
    """
    try:
        df = pd.read_sql(query, conn)
        return df
    finally:
        conn.close()

In [ ]:
def plot_hotness_dotplot(city):
    """绘制极简点图 (sns.stripplot)"""
    df = fetch_neighborhood_hotness_data(city)
    
    if df.empty or len(df) < 3:
        plt.figure(figsize=(10, 6))
        plt.text(0.5, 0.5, f"{city} 有效板块数据不足", ha='center', va='center', fontsize=14)
        plt.axis('off')
        plt.show()
        return

    #为了实现 barh 或水平 stripplot 的 Top 1 在上，需要反转 DataFrame
    df_sorted = df.sort_values(by='weighted_attention', ascending=True)

    # --- 1. 颜色映射逻辑 ---
    norm = plt.Normalize(df_sorted['weighted_attention'].min(), df_sorted['weighted_attention'].max())
    cmap = plt.get_cmap('Oranges') 
    colors = cmap(norm(df_sorted['weighted_attention']))

    # --- 2. 开始绘图 ---
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # 绘制水平 Stripplot (点图)
    stripplot = sns.stripplot(
        x='weighted_attention', 
        y='neighborhood', 
        data=df_sorted,
        size=15, 
        edgecolor='#333333',
        linewidth=1,
        alpha=0.8,
        ax=ax
    )

    from matplotlib.collections import PathCollection
    for artist in ax.collections:
        if isinstance(artist, PathCollection):
            artist.set_facecolors(colors)

    # --- 3. 标注与修饰 ---
    vals = df_sorted['weighted_attention']
    names = df_sorted['neighborhood']
    for i in range(len(vals)):
        ax.text(
            vals.iloc[i] + (vals.max() * 0.02), 
            i, 
            f'{vals.iloc[i]:.1f}', 
            ha='left', va='center', fontsize=11, fontweight='bold', color='#2C3E50'
        )

    plt.title(f'[{city}] 二手房板块关注度热度排行榜 (极简点图)', fontsize=16, pad=20, fontweight='bold')
    plt.xlabel('加权平均关注度 (基于小区房源量加权)', fontsize=12, labelpad=10)
    plt.ylabel('板块名称', fontsize=12)
    
    # 细节美化：
    ax.grid(axis='x', linestyle='--', alpha=0.3)
    ax.tick_params(axis='y', labelsize=12)
    sns.despine(left=True, bottom=True)
    ax.set_xlim(vals.min() * 0.9, vals.max() * 1.15)

    plt.tight_layout()
    plt.show()

In [4]:
def render_interactive_dashboard():
    """构建交互式下拉组件"""
    cities = fetch_cities()
    if not cities:
        print("未获取到城市列表，请检查数据。")
        return
    
    default_city = '北京' if '北京' in cities else cities[0]
    
    city_dropdown = widgets.Dropdown(
        options=cities,
        value=default_city,
        description='📍 分析城市:',
        layout={'width': '250px'}
    )
    
    plot_output = widgets.Output()

    def on_city_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            with plot_output:
                clear_output(wait=True)
                plot_hotness_dotplot(change['new'])

    city_dropdown.observe(on_city_change)

    # 首次展示
    display(city_dropdown)
    display(plot_output)
    
    with plot_output:
        plot_hotness_dotplot(default_city)

In [5]:
if __name__ == "__main__":
    # 执行渲染
    render_interactive_dashboard()

Dropdown(description='📍 分析城市:', index=4, layout=Layout(width='250px'), options=('上海', '东莞', '中山', '佛山', '北京', …

Output()